In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType

# Initialize the SparkSession 

In [2]:
spark = SparkSession.builder \
    .appName("PortoTripsAnalysis") \
    .getOrCreate()

In [3]:
spark

In [4]:
df = spark.read.csv("s3a://taasim/raw/porto-trips/train.csv", header=True)

In [5]:
df.show(10)

+-------------------+---------+-----------+------------+--------+----------+--------+------------+--------------------+
|            TRIP_ID|CALL_TYPE|ORIGIN_CALL|ORIGIN_STAND| TAXI_ID| TIMESTAMP|DAY_TYPE|MISSING_DATA|            POLYLINE|
+-------------------+---------+-----------+------------+--------+----------+--------+------------+--------------------+
|1372636858620000589|        C|       NULL|        NULL|20000589|1372636858|       A|       False|[[-8.618643,41.14...|
|1372637303620000596|        B|       NULL|           7|20000596|1372637303|       A|       False|[[-8.639847,41.15...|
|1372636951620000320|        C|       NULL|        NULL|20000320|1372636951|       A|       False|[[-8.612964,41.14...|
|1372636854620000520|        C|       NULL|        NULL|20000520|1372636854|       A|       False|[[-8.574678,41.15...|
|1372637091620000337|        C|       NULL|        NULL|20000337|1372637091|       A|       False|[[-8.645994,41.18...|
|1372636965620000231|        C|       NU

In [6]:
df.describe()

DataFrame[summary: string, TRIP_ID: string, CALL_TYPE: string, ORIGIN_CALL: string, ORIGIN_STAND: string, TAXI_ID: string, TIMESTAMP: string, DAY_TYPE: string, MISSING_DATA: string, POLYLINE: string]

In [7]:
df.select("TRIP_ID", "TAXI_ID", "POLYLINE").show(5)

+-------------------+--------+--------------------+
|            TRIP_ID| TAXI_ID|            POLYLINE|
+-------------------+--------+--------------------+
|1372636858620000589|20000589|[[-8.618643,41.14...|
|1372637303620000596|20000596|[[-8.639847,41.15...|
|1372636951620000320|20000320|[[-8.612964,41.14...|
|1372636854620000520|20000520|[[-8.574678,41.15...|
|1372637091620000337|20000337|[[-8.645994,41.18...|
+-------------------+--------+--------------------+
only showing top 5 rows



# Parsing the Coordinates

In [8]:
# df.select("POLYLINE").show(1,truncate=False)

In [9]:
# Define the schema for the GPS points
gps_schema = ArrayType(ArrayType(DoubleType()))
# Convert the string column to an actual Array of coordinates
df_parsed = df.withColumn("coords", F.from_json(F.col("POLYLINE"), gps_schema))
# Explode the list so each GPS ping becomes its own row
df_exploded = df_parsed.withColumn("point", F.explode(F.col("coords")))

In [10]:
# df_exploded.show(5,truncate=False)

# The Transformation Math

In [11]:
P_LON_MIN, P_LON_MAX = -8.7, -8.5
P_LAT_MIN, P_LAT_MAX = 41.1, 41.2

C_LON_MIN, C_LON_MAX = -7.8, -7.4
C_LAT_MIN, C_LAT_MAX = 33.4, 33.7

In [12]:
# Apply Transformation for both Lon and Lat
df_final = df_exploded.withColumn("cas_lon", 
    F.lit(C_LON_MIN) + ( (F.col("point")[0] - P_LON_MIN) / (P_LON_MAX - P_LON_MIN) ) * (C_LON_MAX - C_LON_MIN)
).withColumn("cas_lat", 
    F.lit(C_LAT_MIN) + ( (F.col("point")[1] - P_LAT_MIN) / (P_LAT_MAX - P_LAT_MIN) ) * (C_LAT_MAX - C_LAT_MIN)
)

In [13]:
# Clean up: Drop the raw intermediate columns to keep it tidy
df_final = df_final.select("TRIP_ID", "TAXI_ID", "cas_lon", "cas_lat")

In [14]:
df_final.show(5,truncate=False)

+-------------------+--------+------------------+------------------+
|TRIP_ID            |TAXI_ID |cas_lon           |cas_lat           |
+-------------------+--------+------------------+------------------+
|1372636858620000589|20000589|-7.637286000000002|33.524236         |
|1372636858620000589|20000589|-7.636998000000001|33.524128         |
|1372636858620000589|20000589|-7.640652000000002|33.52753          |
|1372636858620000589|20000589|-7.644306000000003|33.531444999999984|
|1372636858620000589|20000589|-7.647906000000002|33.533119         |
+-------------------+--------+------------------+------------------+
only showing top 5 rows



# Mapping Coordinates to Neighborhoods

In [15]:
# Load the Casablanca Zone Mapping (Metadata)
zones_df = spark.read.csv("s3a://taasim/metadata/zone_mapping.csv", header=True, inferSchema=True)

In [16]:
zones_df.show(5,truncate=False)

+-------+------------+-------+-------+-------+-------+
|zone_id|zone_name   |lon_min|lon_max|lat_min|lat_max|
+-------+------------+-------+-------+-------+-------+
|1      |Sidi Belyout|-7.625 |-7.595 |33.59  |33.62  |
|2      |Maarif      |-7.66  |-7.615 |33.565 |33.595 |
|3      |Anfa        |-7.71  |-7.66  |33.585 |33.63  |
|4      |Hay Hassani |-7.73  |-7.67  |33.53  |33.585 |
|5      |Mers Sultan |-7.625 |-7.595 |33.555 |33.59  |
+-------+------------+-------+-------+-------+-------+
only showing top 5 rows



In [17]:
enriched_df = df_final.join(
    zones_df,
    (df_final.cas_lon >= zones_df.lon_min) & 
    (df_final.cas_lon <= zones_df.lon_max) & 
    (df_final.cas_lat >= zones_df.lat_min) & 
    (df_final.cas_lat <= zones_df.lat_max),
    how="left"
).fillna({"zone_id" : -1}) # 'out_of_bounds' as -1

In [18]:
enriched_df.describe()

DataFrame[summary: string, TRIP_ID: string, TAXI_ID: string, cas_lon: string, cas_lat: string, zone_id: string, zone_name: string, lon_min: string, lon_max: string, lat_min: string, lat_max: string]

In [19]:
final_output = enriched_df.select(
    "TRIP_ID", 
    "TAXI_ID", 
    "cas_lon", 
    "cas_lat", 
    "zone_id", 
    "zone_name"
)

In [20]:
final_output.show(10)

+-------------------+--------+-------------------+------------------+-------+---------+
|            TRIP_ID| TAXI_ID|            cas_lon|           cas_lat|zone_id|zone_name|
+-------------------+--------+-------------------+------------------+-------+---------+
|1372636858620000589|20000589| -7.637286000000002|         33.524236|      6|Ain Chock|
|1372636858620000589|20000589| -7.636998000000001|         33.524128|      6|Ain Chock|
|1372636858620000589|20000589| -7.640652000000002|          33.52753|      6|Ain Chock|
|1372636858620000589|20000589| -7.644306000000003|33.531444999999984|      6|Ain Chock|
|1372636858620000589|20000589| -7.647906000000002|         33.533119|      6|Ain Chock|
|1372636858620000589|20000589| -7.653360000000002|         33.534334|      6|Ain Chock|
|1372636858620000589|20000589| -7.654746000000002|         33.534091|      6|Ain Chock|
|1372636858620000589|20000589| -7.660452000000002| 33.53562999999999|     -1|     NULL|
|1372636858620000589|20000589|-7

# Validation & Profiling

In [21]:
%pip install folium

Note: you may need to restart the kernel to use updated packages.


In [22]:
import folium
from folium.plugins import HeatMap

In [23]:
# # Sample 10k points for the plot
# sample_data = df_final.select("cas_lat", "cas_lon").sample(0.01).limit(10000).collect()

# # Create Casablanca Map
# m = folium.Map(location=[33.5731, -7.5898], zoom_start=12)
# HeatMap([[p.cas_lat, p.cas_lon] for p in sample_data]).add_to(m)
# m.save("docs/casablanca-coordinate-validation.png")

In [ ]:
%pip install pyrosm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 5.3 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 3.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pyrobuf-0.9.3-cp311-cp311-linux_x86_64.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.5/342.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 21.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 1.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.5/32.5 MB 11.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 13.4 MB/s eta 0:00:0000:0100:01
  Created wheel for pyrosm: filename=pyrosm-0.6.2-cp311-cp311-linux_x86_6

In [1]:
from pyrosm import OSM

# Initialize the reader with our fresh Morocco snapshot
osm = OSM(r"../data/morocco-260331.osm.pbf")

roads = osm.get_network(network_type="driving")

# 2. Filter the GeoDataFrame using standard Pandas logic
# This matches your requirement for primary, secondary, etc.
target_highways = ["primary", "secondary", "tertiary", "residential"]
roads = roads[roads["highway"].isin(target_highways)]

C_LON_MIN, C_LON_MAX = -7.8, -7.4
C_LAT_MIN, C_LAT_MAX = 33.4, 33.7

roads = roads.cx[C_LON_MIN:C_LON_MAX, C_LAT_MIN:C_LAT_MAX]

# Plot the roads!
roads.plot(figsize=(10, 10), color="gray", alpha=0.5)

ModuleNotFoundError: No module named 'pyrosm'